In [14]:
import pandas as pd
Streak = 2

mite_data = pd.read_csv('mite_data.csv')
mite_data["status_num"] = mite_data["status"].map({"alive": 0, "dead": 1})

mite_data.head()


mite_ids = mite_data['mite ID'].unique()
print(mite_ids)

# get mite_0020 data
mite_0020_data = mite_data[mite_data['mite ID'] == 'mite_0020']

mite_0020_data

['mite_0001' 'mite_0002' 'mite_0003' 'mite_0004' 'mite_0005' 'mite_0006'
 'mite_0007' 'mite_0008' 'mite_0009' 'mite_0010' 'mite_0011' 'mite_0012'
 'mite_0013' 'mite_0014' 'mite_0015' 'mite_0017' 'mite_0018' 'mite_0019'
 'mite_0020' 'mite_0021' 'mite_0022' 'mite_0023' 'mite_0024' 'mite_0025'
 'mite_0026' 'mite_0027' 'mite_0028' 'mite_0029' 'mite_0030' 'mite_0031'
 'mite_0032' 'mite_0033' 'mite_0034' 'mite_0035' 'mite_0036' 'mite_0037'
 'mite_0038' 'mite_0039' 'mite_0040' 'mite_0041' 'mite_0042' 'mite_0043'
 'mite_0044' 'mite_0045' 'mite_0046' 'mite_0047' 'mite_0048' 'mite_0049'
 'mite_0050' 'mite_0051' 'mite_0052' 'mite_0053' 'mite_0054' 'mite_0055'
 'mite_0056' 'mite_0057' 'mite_0058']


,mite ID,zone ID,status,max diff,local diff,recording,is_dead,streak_id,first_dead_streak,status_num
144,mite_0020,alive,dead,17.0,16.333333,1,True,1,NaN,1
145,mite_0020,alive,alive,74.0,59.333333,2,False,2,NaN,0
146,mite_0020,alive,dead,11.0,10.666667,3,True,3,NaN,1
147,mite_0020,alive,alive,53.0,49.000000,4,False,4,NaN,0
148,mite_0020,alive,dead,10.0,10.000000,5,True,5,NaN,1
149,mite_0020,alive,alive,50.0,42.000000,6,False,6,NaN,0
150,mite_0020,alive,dead,11.0,10.333333,7,True,7,NaN,1
151,mite_0020,alive,dead,11.0,11.000000,8,True,7,NaN,1


In [15]:
# get the mite statuses
mite_0020_statuses = mite_0020_data['status_num']

# apply rolling function to get streak
mite_0020_data["rolling sum"] = mite_0020_data["status_num"].rolling(window=Streak).sum()



mask = mite_0020_data["rolling sum"] == Streak
if mask.any():
    row = mite_0020_data.loc[mask].iloc[0]              # get the first matching row
    rec_num = row["recording"]              # value from the recording column
    print("Recording number where rolling sum first =", Streak, ":", rec_num)
    print("start of streak at:", rec_num - Streak + 1)
else:
    print("No rolling sum of", Streak, "found")


Recording number where rolling sum first = 2 : 8
start of streak at: 7


C:\Users\mateo\AppData\Local\Temp\ipykernel_17392\3523563680.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mite_0020_data["rolling sum"] = mite_0020_data["status_num"].rolling(window=Streak).sum()


In [16]:
# Assuming you have a class or context where post_process_mite_data is defined,
# and you want to use it on the whole mite_data DataFrame.

# If post_process_mite_data is a method of a class, instantiate the class first.
# For demonstration, let's assume it's a standalone function (remove 'self' from definition if needed).

def post_process_mite_data(mites_data, streak):
    # map status to numbers
    mites_data.loc[:, "status_num"] = mites_data["status"].map({"alive": 0, "dead": 1})

    # helper to get streak start for one group
    def get_streak_start(series):
        rolling_sum = series.rolling(window=streak).sum()
        mask = rolling_sum == streak
        if mask.any():
            idx = mask.idxmax()  # first index where streak occurs
            rec_num = mites_data.loc[idx, "recording"]
            return rec_num - streak + 1
        else:
            return None

    # group by mite_ID and compute streak_start per mite
    streak_starts = mites_data.groupby("mite ID")["status_num"].apply(get_streak_start)

    # map the streak_start back to the original DataFrame
    mites_data["streak_start"] = mites_data["mite ID"].map(streak_starts)

    return mites_data


processed_mite_data = post_process_mite_data(mite_data, Streak)
processed_mite_data

,mite ID,zone ID,status,max diff,local diff,recording,is_dead,streak_id,first_dead_streak,status_num,streak_start
0,mite_0001,dead,dead,11.0,9.666667,1,True,1,1.0,1,1.0
1,mite_0001,dead,dead,11.0,10.666667,2,True,1,1.0,1,1.0
2,mite_0001,dead,dead,11.0,10.333333,3,True,1,1.0,1,1.0
3,mite_0001,dead,dead,12.0,10.000000,4,True,1,1.0,1,1.0
4,mite_0001,dead,dead,10.0,10.000000,5,True,1,1.0,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...
451,mite_0058,dead,dead,13.0,11.666667,4,True,1,1.0,1,1.0
452,mite_0058,dead,dead,14.0,12.333333,5,True,1,1.0,1,1.0
453,mite_0058,dead,dead,14.0,12.000000,6,True,1,1.0,1,1.0
454,mite_0058,dead,dead,12.0,11.666667,7,True,1,1.0,1,1.0
